In [10]:
# Cell 1: Imports & Configuration
import os
import sys
import json
import pickle
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import silhouette_score, adjusted_rand_score, adjusted_mutual_info_score
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_from_disk  # [新增] 读取 arrow 文件

# 假设当前目录是项目根目录
sys.path.append(os.getcwd())

# [修改] 数据源指向 processed_datasets
DATA_DIR = "Datasets/processed_datasets" 
PROPERTIES_PATH = "Datasets/processed_datasets/dataset_properties.json"
RESULT_CSV = "encoder_benchmark_results.csv"

# 全局配置
CONFIG = {
    "context_len": 512,  # 编码器输入长度 (即历史窗口长度)
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "batch_size": 64,
    "seed": 2025,
    "max_samples_per_dataset": 200 # [新增] 每个子数据集最大采样数
}

# 设置随机种子
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG['seed'])
print(f"✅ 环境配置完成。设备: {CONFIG['device']}")
print(f"📂 数据源: {DATA_DIR}")

✅ 环境配置完成。设备: cuda
📂 数据源: Datasets/processed_datasets


In [11]:
# Cell 2: Metadata Manager (Case-Insensitive Fix)

class MetadataManager:
    def __init__(self, properties_path):
        with open(properties_path, 'r', encoding='utf-8') as f:
            self.props = json.load(f)
            
    def _get_info(self, name):
        """内部辅助函数：标准化名称并查找属性"""
        # [核心修复] 强制转换为小写进行匹配，因为 json 中的 key 都是小写的
        key = name.lower()
        
        # 1. 直接匹配
        info = self.props.get(key)
        
        if not info:
            # 2. 尝试去掉后缀匹配 (比如 _short, _medium)
            # 对小写后的 key 进行分割
            parts = key.split("_")
            if len(parts) > 1:
                base_name = "_".join(parts[:-1])
                info = self.props.get(base_name)
                
        # 3. 再次尝试：有时候数据集名是 'dataset_freq' 格式，json 里只有 'dataset'
        if not info and len(parts) > 1:
             # 尝试再去掉一级后缀 (针对 name_freq_term 这种情况)
             base_name_2 = "_".join(parts[:-2])
             info = self.props.get(base_name_2)

        return info if info else {"domain": "Unknown", "frequency": "Unknown"}

    def get_domain(self, dataset_name):
        """查询单个数据集的领域"""
        info = self._get_info(dataset_name)
        return info.get("domain", "Unknown")

    def get_labels(self, dataset_names):
        """
        根据数据集名称列表，返回多维度的标签 (Domain, Frequency)
        """
        domains = []
        freqs = []
        
        for name in dataset_names:
            info = self._get_info(name)
            domains.append(info.get("domain", "Unknown"))
            freqs.append(info.get("frequency", "Unknown"))
            
        return np.array(domains), np.array(freqs)

meta_manager = MetadataManager(PROPERTIES_PATH)
print("✅ 元数据管理器已重新加载 (已修复大小写匹配问题)")

✅ 元数据管理器已重新加载 (已修复大小写匹配问题)


In [12]:
# Cell 3: Data Loader (From Processed Arrow Files)

def process_tensor(arr, target_len):
    """
    复刻 dataset.py 中的 _process_tensor: Numpy -> Tensor -> Pad/Truncate
    """
    t_np = np.array(arr, dtype=np.float32).reshape(-1)
    t = torch.from_numpy(t_np).unsqueeze(0) # (1, L)
    
    # 这里的逻辑是：如果不足 target_len 则在左侧补0 (History模式)
    # 如果超过则取最后 target_len (但在滑窗逻辑中，我们通常切好的就是 target_len 或更短)
    if t.shape[1] < target_len:
        pad = (target_len - t.shape[1], 0) 
        t = torch.nn.functional.pad(t, pad)
    else:
        t = t[:, -target_len:]
    return t

def load_all_sequences(data_dir, context_len, meta_manager, max_samples=100):
    """
    从 processed_datasets 加载原始序列，并构造成编码器输入
    
    参数:
      - max_samples: 每个数据集限制的最大样本数
    """
    # 1. 扫描所有子数据集目录
    if not os.path.exists(data_dir):
        raise FileNotFoundError(f"目录不存在: {data_dir}")
        
    dataset_dirs = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    dataset_dirs.sort()
    
    all_norm_tensors = []
    all_raw_tensors = []
    all_scales = []
    all_ds_names = []
    
    skipped_domain_count = 0
    loaded_ds_count = 0
    
    print(f"📂 开始扫描 {len(dataset_dirs)} 个数据集 (Max samples/ds: {max_samples})...")
    
    for ds_name in tqdm(dataset_dirs):
        # --- A. 领域过滤 ---
        domain = meta_manager.get_domain(ds_name)
        if domain == "Other":
            skipped_domain_count += 1
            continue
            
        ds_path = os.path.join(data_dir, ds_name)
        current_ds_samples = 0
        
        try:
            # --- B. 加载 HuggingFace Dataset ---
            # 这里的 dataset 通常包含 'start', 'target', 'feat_...' 等字段
            hf_ds = load_from_disk(ds_path)
            if len(hf_ds) == 0: continue
                
            # --- C. 遍历序列并滑窗构造样本 ---
            # 既然要求步长 = context_len (无重叠)，我们对每个长序列进行切分
            for entry in hf_ds:
                if current_ds_samples >= max_samples:
                    break
                    
                full_series = entry['target'] # list or numpy
                series_len = len(full_series)
                
                # 步长 = 上下文长度
                stride = context_len 
                
                # 开始切片
                for start_idx in range(0, series_len, stride):
                    if current_ds_samples >= max_samples:
                        break
                        
                    end_idx = start_idx + context_len
                    # 获取窗口数据 (如果是最后一段，可能不足 context_len)
                    window_data = full_series[start_idx : min(end_idx, series_len)]
                    
                    # 只有当包含有效数据时才处理
                    if len(window_data) > 0:
                        # --- D. 预处理 (复刻 dataset.py) ---
                        # 1. 转 Tensor & Pad
                        hist_t = process_tensor(window_data, context_len)
                        
                        # 2. Local Scaling
                        scale = torch.mean(torch.abs(hist_t)).item()
                        scale = max(scale, 1e-6)
                        
                        # 3. Normalize
                        hist_norm = hist_t / scale
                        
                        # 收集
                        all_norm_tensors.append(hist_norm)
                        all_raw_tensors.append(hist_t)
                        all_scales.append(scale)
                        all_ds_names.append(ds_name)
                        
                        current_ds_samples += 1
            
            if current_ds_samples > 0:
                loaded_ds_count += 1
                
        except Exception as e:
            # print(f"⚠️ 加载 {ds_name} 失败: {e}")
            pass
            
    # 堆叠结果
    if not all_norm_tensors:
        raise ValueError("未生成任何样本！请检查数据路径或过滤条件。")
        
    print(f"📊 加载完成:")
    print(f"   - 扫描数据集: {len(dataset_dirs)}")
    print(f"   - 过滤(Other): {skipped_domain_count}")
    print(f"   - 实载数据集: {loaded_ds_count}")
    print(f"   - 总样本数: {len(all_norm_tensors)}")
        
    return (
        torch.cat(all_norm_tensors, dim=0), 
        torch.cat(all_raw_tensors, dim=0),
        np.array(all_scales),
        np.array(all_ds_names)
    )

# 执行加载
X_norm, X_raw, X_scales, ds_names_list = load_all_sequences(
    DATA_DIR, 
    CONFIG['context_len'], 
    meta_manager, 
    max_samples=CONFIG['max_samples_per_dataset']
)
print(f"✅ 样本构造完成。Shape: {X_norm.shape}")

📂 开始扫描 327 个数据集 (Max samples/ds: 200)...


100%|██████████| 327/327 [00:07<00:00, 44.16it/s] 

📊 加载完成:
   - 扫描数据集: 327
   - 过滤(Other): 128
   - 实载数据集: 198
   - 总样本数: 26287
✅ 样本构造完成。Shape: torch.Size([26287, 512])


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pywt  # 需要安装 PyWavelets
from scipy.stats import skew, kurtosis

# 确保继承自您之前定义的 BaseEncoder
class BaseEncoder(nn.Module):
    def __init__(self): super().__init__()
    def encode(self, x): raise NotImplementedError

# =========================================================
# 1. FFT Encoder (频谱分析)
# =========================================================
class FFTEncoder(BaseEncoder):
    def __init__(self, output_dim=128):
        super().__init__()
        self.name = "FFT(Spectral)"
        self.output_dim = output_dim
        # 这是一个无参数方法，不需要加载权重
        self.model_loaded = True 

    def encode(self, x):
        """
        x: (Batch, Seq_Len)
        """
        # 转为 CPU numpy 处理或直接用 Torch FFT
        if isinstance(x, np.ndarray): x = torch.from_numpy(x)
        device = x.device
        
        # 1. 执行实数 FFT
        # rfft 返回复数张量，形状 (B, L/2 + 1)
        fft_coeffs = torch.fft.rfft(x, dim=-1)
        
        # 2. 获取幅值 (Amplitude)
        # 我们只关心能量大小，忽略相位信息（对于相似性检索，相位偏移通常不那么重要）
        # 取模
        fft_amp = fft_coeffs.abs()
        
        # 3. 维度调整 (截断或补零到 output_dim)
        B, F_len = fft_amp.shape
        
        if F_len >= self.output_dim:
            # 截取低频部分 (通常包含最多信息)
            emb = fft_amp[:, :self.output_dim]
        else:
            # 补零
            padding = torch.zeros(B, self.output_dim - F_len, device=device)
            emb = torch.cat([fft_amp, padding], dim=1)
            
        # 4. 归一化 (L2)
        return F.normalize(emb.float(), p=2, dim=1).cpu()

# =========================================================
# 2. Statistical Encoder (统计特征)
# =========================================================
class StatisticsEncoder(BaseEncoder):
    def __init__(self, output_dim=128):
        super().__init__()
        self.name = "Stats(CATCH22-like)"
        self.output_dim = output_dim
        self.model_loaded = True
        
        # 为了将低维统计特征映射到高维 (128)，我们使用一个固定的随机正交矩阵
        # 这是一个常用的技巧 (Random Projection)，可以保持距离关系
        # 假设我们提取 9 个基础统计特征
        self.num_base_features = 9
        # 固定随机种子保证复现性
        rng = torch.Generator()
        rng.manual_seed(42)
        self.projection = nn.Linear(self.num_base_features, output_dim, bias=False)
        # 冻结投影矩阵，使其仅仅是一个数学变换
        for param in self.projection.parameters():
            param.requires_grad = False
            # 使用正交初始化
            nn.init.orthogonal_(param)

    def _get_stats_torch(self, x):
        """完全使用 PyTorch 计算统计量以支持 GPU 加速"""
        # x: (B, L)
        device = x.device
        B, L = x.shape
        
        # 1. Mean
        mean = x.mean(dim=1, keepdim=True)
        
        # 2. Std
        std = x.std(dim=1, keepdim=True) + 1e-6
        
        # 3. Skewness (偏度)
        centered = x - mean
        skew = (centered ** 3).mean(dim=1, keepdim=True) / (std ** 3)
        
        # 4. Kurtosis (峰度)
        kurt = (centered ** 4).mean(dim=1, keepdim=True) / (std ** 4)
        
        # 5. Max
        max_val = x.max(dim=1, keepdim=True).values
        
        # 6. Min
        min_val = x.min(dim=1, keepdim=True).values
        
        # 7. Zero Crossing Rate (过零率 - 归一化后的)
        # 先归一化到 0 均值
        norm_x = (x - mean)
        # 计算相邻符号变化的次数
        zcr = ((norm_x[:, :-1] * norm_x[:, 1:]) < 0).float().mean(dim=1, keepdim=True)
        
        # 8. Autocorrelation (Lag-1)
        # 简单的自相关计算
        lag1_num = ((x[:, :-1] - mean) * (x[:, 1:] - mean)).mean(dim=1, keepdim=True)
        acf_1 = lag1_num / (std ** 2)
        
        # 9. Slope (简单线性回归的斜率，表示趋势)
        # t = torch.arange(L, device=device, dtype=torch.float32)
        # 这里简化为首尾差分，近似趋势
        trend = (x[:, -1] - x[:, 0]).unsqueeze(-1) / L

        # 拼接所有特征 (B, 9)
        features = torch.cat([mean, std, skew, kurt, max_val, min_val, zcr, acf_1, trend], dim=1)
        
        # 处理可能的 NaN (例如常数序列导致 std=0)
        features = torch.nan_to_num(features, nan=0.0)
        return features

    def encode(self, x):
        if isinstance(x, np.ndarray): x = torch.from_numpy(x)
        
        # 计算基础特征
        base_feats = self._get_stats_torch(x.float())
        
        # 投影到目标维度 (128)
        # 注意：需要把 projection 移动到通过一个 device
        if self.projection.weight.device != x.device:
            self.projection = self.projection.to(x.device)
            
        emb = self.projection(base_feats)
        
        return F.normalize(emb, p=2, dim=1).cpu()

# =========================================================
# 3. Wavelet Encoder (小波变换)
# =========================================================
class WaveletEncoder(BaseEncoder):
    def __init__(self, output_dim=128, wavelet='db4', level=3):
        super().__init__()
        self.name = "Wavelet(DWT)"
        self.output_dim = output_dim
        self.wavelet = wavelet
        self.level = level
        self.model_loaded = True

    def encode(self, x):
        """
        小波变换通常在 CPU 上使用 pywt 库运行较快且方便
        """
        if isinstance(x, torch.Tensor):
            x_np = x.detach().cpu().numpy()
        else:
            x_np = x
            
        B, L = x_np.shape
        embeddings = []
        
        # PyWavelets 不支持 Batch 并行，需要循环或使用高级包装
        # 这里为了简单清晰使用循环，由于是提取特征，速度通常可接受
        for i in range(B):
            ts = x_np[i]
            # 1. 离散小波分解
            # coeffs 返回 [cA_n, cD_n, cD_n-1, ..., cD_1]
            # cA: 近似系数 (低频/趋势), cD: 细节系数 (高频/噪声)
            try:
                coeffs = pywt.wavedec(ts, self.wavelet, level=self.level)
            except ValueError:
                # 如果序列太短无法分解到指定层级，自动调整
                coeffs = pywt.wavedec(ts, self.wavelet, level=None)
            
            # 2. 特征提取策略：拼接所有系数并截断/Pad
            # 也可以提取每一层的能量 (Energy) 和 熵 (Entropy) 作为特征，这里选择直接保留系数结构
            flat_coeffs = np.concatenate(coeffs)
            
            if len(flat_coeffs) >= self.output_dim:
                feat = flat_coeffs[:self.output_dim]
            else:
                feat = np.pad(flat_coeffs, (0, self.output_dim - len(flat_coeffs)), 'constant')
            
            embeddings.append(feat)
            
        emb_tensor = torch.tensor(np.array(embeddings), dtype=torch.float32)
        
        return F.normalize(emb_tensor, p=2, dim=1)

# =========================================================
# 初始化并添加到列表
# =========================================================
# 假设 output_dim 统一为 128 以便于和 TimesFM/UniTS/Moirai 对比
# (TimesFM 实际上输出 1280，如果你要和它做 Cosine Similarity，
#  最好将这些数学特征也投影到 1280，或者只取 TimesFM 的前 128)
# 为了可视化 (t-SNE) 和聚类，维度不一致其实没关系，只要之后用 t-SNE 降到 2维即可。
# 但计算距离矩阵时维度必须一致吗？如果不一致，不同 encoder 之间无法直接算距离，
# 但同一 encoder 内部样本的距离分布是可以比较的。
# 这里默认保持 128，因为你的 encoder_ana.ipynb 可能是单独对每个 encoder 跑指标。

math_encoders = [
    FFTEncoder(output_dim=128),
    StatisticsEncoder(output_dim=128),
    WaveletEncoder(output_dim=128)
]

# 将它们加入到你的 encoders 列表
# encoders.extend(math_encoders) 
print(f"📊 已构建数学编码器: {[e.name for e in math_encoders]}")

📊 已构建数学编码器: ['FFT(Spectral)', 'Stats(CATCH22-like)', 'Wavelet(DWT)']


In [14]:
# Cell 4: Encoder Interface & Implementations (Fixed: Moirai Max-Patch Padding)

import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import re
import argparse
import inspect
from typing import Union

# 确保能导入项目模块
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# =========================================================
# 0. BaseEncoder Interface
# =========================================================
class BaseEncoder(nn.Module):
    def __init__(self): super().__init__()
    def encode(self, x): raise NotImplementedError

# =========================================================
# 1. Checkpoint Paths
# =========================================================
UNITS_CKPT_PATH = "checkpoints/units_x128_pretrain_checkpoint.pth" 
TIMESFM_CKPT_PATH = "checkpoints/timesfm-2.5-200m-pytorch"
MOIRAI_CKPT_PATH = "checkpoints/moirai-1.1-R-base" 

# =========================================================
# 2. UniTS Encoder (保持不变)
# =========================================================
class UnitsEncoderWrapper(BaseEncoder):
    def __init__(self, ckpt_path, context_len=96, device='cuda'):
        super().__init__()
        self.device = device
        self.name = "UniTS"
        self.model_loaded = False
        print(f"🧠 [UniTS] Initializing... Loading weights: {os.path.basename(ckpt_path)}")
        
        current_dir = os.getcwd()
        units_path = os.path.join(current_dir, 'UniTS-main')
        if os.path.exists(units_path) and units_path not in sys.path:
            sys.path.insert(0, units_path)

        try:
            from models.UniTS_zeroshot import Model as UniTSZeroShotModel
            d_model = 128
            match = re.search(r'_x(\d+)_', os.path.basename(ckpt_path))
            if match: d_model = int(match.group(1))
            
            args = argparse.Namespace(
                d_model=d_model, n_heads=8 if d_model <= 128 else 16, e_layers=3,
                patch_len=16, stride=16, dropout=0.1, prompt_num=10,
                right_prob=0.5, min_mask_ratio=0.5, max_mask_ratio=0.8, 
                min_keep_ratio=0.5, prompt_len=10
            )
            dummy_config = [['Generic_Task', {'task_name': 'classification', 'dataset': 'Generic', 'enc_in': 1, 'num_class': 1, 'seq_len': context_len, 'label_len': 0, 'pred_len': 0}]]
            self.model = UniTSZeroShotModel(args, dummy_config, pretrain=False)
            
            try:
                checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
            except TypeError:
                checkpoint = torch.load(ckpt_path, map_location=device)

            state_dict = checkpoint.get('state_dict', checkpoint)
            if 'student' in state_dict: state_dict = state_dict['student']
            new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items() if 'forecast_head' not in k and 'cls_head' not in k}
            
            self.model.load_state_dict(new_state_dict, strict=False)
            self.model.to(device)
            self.model.eval()
            self.model_loaded = True
            print(f"✅ [UniTS] Loaded successfully (d_model={d_model})")
        except Exception as e:
            print(f"⚠️ [UniTS] Load failed: {e}")

    def encode(self, x):
        if not self.model_loaded: return torch.zeros(x.shape[0], 128).to(self.device)
        if isinstance(x, np.ndarray): x = torch.from_numpy(x)
        x = x.float().to(self.device)
        if x.dim() == 2: x = x.unsqueeze(-1)
        
        with torch.no_grad():
            x_enc, _, _, n_vars, _ = self.model.tokenize(x)
            if hasattr(self.model, 'prompt_token'):
                prefix_prompt = self.model.prompt_token.repeat(x_enc.shape[0], n_vars, 1, 1)
                task_prompt = self.model.cls_token.repeat(x_enc.shape[0], n_vars, 1, 1)
            else:
                prefix_prompt = torch.zeros(x_enc.shape[0], n_vars, 10, self.model.args.d_model, device=self.device)
                task_prompt = torch.zeros(x_enc.shape[0], n_vars, 1, self.model.args.d_model, device=self.device)
            x_enc = torch.reshape(x_enc, (-1, n_vars, x_enc.shape[-2], x_enc.shape[-1]))
            x_enc = x_enc + self.model.position_embedding(x_enc)
            x_final = torch.cat((prefix_prompt, x_enc, task_prompt), dim=2)
            enc_out = self.model.backbone(x_final, prefix_len=10, seq_len=x_final.shape[-2]-10)
            valid_tokens = enc_out[:, :, 10:-1, :]
            emb = valid_tokens.mean(dim=2).mean(dim=1)
        return F.normalize(emb, p=2, dim=1).cpu()

# =========================================================
# 3. TimesFM Encoder (保持不变)
# =========================================================
class TimesFMEncoderWrapper(BaseEncoder):
    def __init__(self, ckpt_path, context_len=512, device='cuda'):
        super().__init__()
        self.device = torch.device(device)
        self.name = "TimesFM"
        self.model_loaded = False
        print(f"⏳ [TimesFM] Initializing... Loading model: {ckpt_path}")
        
        try:
            from model_zoo.TSFM_src.timesfm.timesfm_2p5 import timesfm_2p5_torch
            self.tfm_wrapper = timesfm_2p5_torch.TimesFM_2p5_200M_torch.from_pretrained(
                ckpt_path,
                local_files_only=True
            )
            self.model = self.tfm_wrapper.model
            self.model.to(self.device)
            self.model.eval()
            self.model_loaded = True
            print(f"✅ [TimesFM] Loaded successfully (Native)")
        except Exception as e:
            print(f"⚠️ [TimesFM] Load failed: {e}")

    def encode(self, x):
        if not self.model_loaded: return torch.zeros(x.shape[0], 1280).to(self.device)
        if isinstance(x, np.ndarray): x = torch.from_numpy(x)
        x = x.float().to(self.device)
        if x.dim() == 3: x = x.squeeze(-1) 
        
        B, L = x.shape
        patch_len = 32
        
        if L % patch_len != 0:
            pad_len = patch_len - (L % patch_len)
            x = F.pad(x, (pad_len, 0)) 
        
        x_patched = x.view(B, -1, patch_len)
        mask_patched = torch.zeros_like(x_patched) # 0=Valid
        
        if L % patch_len != 0:
            pad_len = patch_len - (L % patch_len)
            flat_mask = torch.zeros((B, x.shape[1]), device=self.device)
            flat_mask[:, :pad_len] = 1.0 
            mask_patched = flat_mask.view(B, -1, patch_len)
            
        with torch.no_grad():
            try:
                outputs = self.model(x_patched, mask_patched)
                output_tensor = outputs
                while isinstance(output_tensor, (tuple, list)):
                    if len(output_tensor) > 0: output_tensor = output_tensor[0]
                    else: break
                embeddings = output_tensor[:, -1, :]
            except Exception as e:
                return torch.zeros(B, 1280).to(self.device)
                
        return F.normalize(embeddings, p=2, dim=1).cpu()

# =========================================================
# 4. Moirai Feature Extractor (修复 Config + MaxPatch)
# =========================================================
try:
    from uni2ts.model.moirai import MoiraiModule
    from uni2ts.common.torch_util import mask_fill, packed_attention_mask
    
    class MoiraiFeatureExtractor(MoiraiModule):
        def __init__(self, *args, **kwargs):
            # Config Fix: 替换 distr_output 为 Dummy 对象
            if 'distr_output' in kwargs and isinstance(kwargs['distr_output'], dict):
                class DummyDistrOutput:
                    def get_param_proj(self, d_model, patch_sizes):
                        return nn.Linear(d_model, 1)
                    def distribution(self, *args, **kwargs): return None
                kwargs['distr_output'] = DummyDistrOutput()
            super().__init__(*args, **kwargs)

        def forward(self, target, observed_mask, sample_id, time_id, variate_id, prediction_mask, patch_size):
            # 特征提取 Forward，无分布投影
            loc, scale = self.scaler(target, observed_mask * ~prediction_mask.unsqueeze(-1), sample_id, variate_id)
            scaled_target = (target - loc) / scale
            reprs = self.in_proj(scaled_target, patch_size)
            masked_reprs = mask_fill(reprs, prediction_mask, self.mask_encoding.weight)
            reprs = self.encoder(masked_reprs, packed_attention_mask(sample_id), time_id=time_id, var_id=variate_id)
            return reprs

except ImportError:
    print("⚠️ 无法导入 uni2ts，Moirai 模块将不可用。")
    MoiraiFeatureExtractor = None

# =========================================================
# 5. Moirai Wrapper (修复 Patch 维度 32 -> 128)
# =========================================================
class MoiraiEncoderWrapper(BaseEncoder):
    def __init__(self, ckpt_path, context_len=512, device='cuda', patch_size=32):
        super().__init__()
        self.device = torch.device(device)
        self.name = "Moirai"
        self.patch_size = patch_size # 实际使用的 patch size (32)
        self.max_patch_size = 128    # [关键] Moirai Base 模型要求的最大 patch size
        self.model_loaded = False
        
        print(f"⏳ [Moirai] 初始化... 加载权重: {ckpt_path}")
        
        if MoiraiFeatureExtractor is None:
            print("❌ uni2ts 未安装")
            return
        if not os.path.exists(ckpt_path):
            print(f"❌ 路径不存在: {ckpt_path}")
            return

        try:
            self.model = MoiraiFeatureExtractor.from_pretrained(ckpt_path, map_location="cpu")
            self.model.to(self.device)
            self.model.eval()
            self.model_loaded = True
            print(f"✅ [Moirai] 加载成功 (Feature Extractor Mode)")
        except Exception as e:
            print(f"⚠️ [Moirai] 加载失败: {e}")

    def encode(self, x):
        if not self.model_loaded: return torch.zeros(x.shape[0], 128).to(self.device)
        if isinstance(x, np.ndarray): x = torch.from_numpy(x)
        x = x.float().to(self.device)
        if x.dim() == 2: x = x.unsqueeze(-1)
        
        B, L, V = x.shape
        P = self.patch_size       # 32
        MAX_P = self.max_patch_size # 128
        
        # 1. Padding 时间维度 (补齐 32)
        pad_len = 0
        if L % P != 0:
            pad_len = P - (L % P)
            x = F.pad(x, (0, 0, pad_len, 0)) # Pad time dim
        
        # 2. Reshape to Patches: (B, Num_Patches, 32)
        num_patches = x.shape[1] // P
        target_raw = x.squeeze(-1).view(B, num_patches, P) 
        
        # 3. [关键修复] Padding Patch维度 (32 -> 128)
        # Moirai 要求输入维度必须匹配 max_patch_size
        target = F.pad(target_raw, (0, MAX_P - P)) # (B, N, 128)
        
        # 4. 构造 observed_mask
        # 真实数据区域为 1，时间Padding区域为 0，Patch填充区域(32-128)为 0
        observed_mask = torch.ones_like(target_raw, dtype=torch.bool)
        if pad_len > 0:
            flat_mask = torch.ones((B, x.shape[1]), dtype=torch.bool, device=self.device)
            flat_mask[:, :pad_len] = False 
            observed_mask = flat_mask.view(B, num_patches, P)
        
        # Mask 也要 Pad 到 128
        observed_mask = F.pad(observed_mask, (0, MAX_P - P), value=False)

        # 5. IDs
        sample_id = torch.zeros((B, num_patches), dtype=torch.long, device=self.device)
        time_id = torch.arange(num_patches, device=self.device).expand(B, -1)
        variate_id = torch.zeros((B, num_patches), dtype=torch.long, device=self.device)
        prediction_mask = torch.zeros((B, num_patches), dtype=torch.bool, device=self.device)
        
        # Patch Size Tensor (告诉模型这一块只有前32是有效的)
        patch_size_tensor = torch.full((B, num_patches), P, dtype=torch.long, device=self.device)
        
        with torch.no_grad():
            reprs = self.model(
                target=target,
                observed_mask=observed_mask,
                sample_id=sample_id,
                time_id=time_id,
                variate_id=variate_id,
                prediction_mask=prediction_mask,
                patch_size=patch_size_tensor
            )
            emb = reprs.mean(dim=1)
            
        return F.normalize(emb, p=2, dim=1).cpu()

# =========================================================
# 6. Initialization
# =========================================================
encoders = []
if os.path.exists(UNITS_CKPT_PATH):
    e = UnitsEncoderWrapper(UNITS_CKPT_PATH, context_len=CONFIG['context_len'], device=CONFIG['device'])
    if e.model_loaded: encoders.append(e)

if os.path.exists(TIMESFM_CKPT_PATH):
    e = TimesFMEncoderWrapper(TIMESFM_CKPT_PATH, context_len=CONFIG['context_len'], device=CONFIG['device'])
    if e.model_loaded: encoders.append(e)

if os.path.exists(MOIRAI_CKPT_PATH):
    e = MoiraiEncoderWrapper(MOIRAI_CKPT_PATH, context_len=CONFIG['context_len'], device=CONFIG['device'])
    if e.model_loaded: encoders.append(e)

print(f"\n📊 Active Encoders: {[e.name for e in encoders]}")

🧠 [UniTS] Initializing... Loading weights: units_x128_pretrain_checkpoint.pth
✅ [UniTS] Loaded successfully (d_model=128)
⏳ [TimesFM] Initializing... Loading model: checkpoints/timesfm-2.5-200m-pytorch
✅ [TimesFM] Loaded successfully (Native)
⏳ [Moirai] 初始化... 加载权重: checkpoints/moirai-1.1-R-base
Loading weights from local directory
✅ [Moirai] 加载成功 (Feature Extractor Mode)

📊 Active Encoders: ['UniTS', 'TimesFM', 'Moirai']


In [15]:
encoders.extend(math_encoders)
print(f"🚀 所有编码器就绪: {[e.name for e in encoders]}")

🚀 所有编码器就绪: ['UniTS', 'TimesFM', 'Moirai', 'FFT(Spectral)', 'Stats(CATCH22-like)', 'Wavelet(DWT)']


In [16]:
# Cell 5: Perturbation Functions for Robustness Check

def apply_perturbation(x, method="noise", severity=0.1):
    """
    对输入序列 x (B, L) 施加扰动
    """
    x_aug = x.clone()
    B, L = x.shape
    device = x.device
    
    if method == "noise":
        # 加高斯噪声
        noise = torch.randn_like(x_aug) * severity
        x_aug += noise
        
    elif method == "mask":
        # 随机 Mask 掉一部分 (置为0)
        mask_len = int(L * severity)
        start_idx = np.random.randint(0, L - mask_len, size=B)
        for i in range(B):
            x_aug[i, start_idx[i]:start_idx[i]+mask_len] = 0
            
    elif method == "shift":
        # 时间平移 (简单 Roll)
        shift = int(L * severity)
        x_aug = torch.roll(x_aug, shifts=shift, dims=1)
        # 填充平移产生的空洞
        x_aug[:, :shift] = 0 
        
    elif method == "scale":
        # 幅度缩放
        scaler = 1.0 + (torch.rand(B, 1, device=device) - 0.5) * severity * 2
        x_aug *= scaler
        
    return x_aug

print("✅ 扰动函数定义完成")

✅ 扰动函数定义完成


In [17]:
# Cell 6: Evaluation Engine

def evaluate_clustering(embeddings_np, labels):
    """
    计算聚类指标: Silhouette, AMI, ARI
    """
    if len(np.unique(labels)) < 2:
        return np.nan, np.nan, np.nan
        
    sil = silhouette_score(embeddings_np, labels)
    ami = adjusted_mutual_info_score(labels, embeddings_np.argmax(axis=1) if embeddings_np.shape[1] > 2 else labels) # 近似处理
    # 注意：ARI 等指标通常需要聚类结果，这里我们直接用 Silhouette 衡量类内距离/类间距离
    # 为了简化，我们主要关注 Silhouette
    return sil

def evaluate_robustness(encoder, x_original, batch_size=64):
    """
    计算不同扰动下的编码距离
    """
    methods = ["noise", "mask", "shift", "scale"]
    results = {}
    
    # 1. 原始编码
    z_orig = []
    for i in range(0, len(x_original), batch_size):
        batch = x_original[i:i+batch_size]
        z_orig.append(encoder.encode(batch).cpu())
    z_orig = torch.cat(z_orig)
    
    # 2. 扰动编码并计算距离
    for method in methods:
        x_pert = apply_perturbation(x_original, method=method, severity=0.1) # 默认 10% 扰动
        z_pert = []
        for i in range(0, len(x_pert), batch_size):
            batch = x_pert[i:i+batch_size]
            z_pert.append(encoder.encode(batch).cpu())
        z_pert = torch.cat(z_pert)
        
        # 计算 Cosine Distance (1 - Cosine Similarity)
        cos_sim = torch.nn.functional.cosine_similarity(z_orig, z_pert)
        avg_dist = 1.0 - cos_sim.mean().item()
        results[f"dist_{method}"] = avg_dist
        
    return results

print("✅ 评估引擎准备就绪")

✅ 评估引擎准备就绪


In [18]:
# Cell 7: Main Benchmark Loop (Debug & Robust Version)

results_table = []

# 对齐检查
if len(ds_names_list) != len(X_norm):
    print(f"⚠️ 警告: 数据({len(X_norm)})与名称({len(ds_names_list)})长度不一致! 截断对齐...")
    min_len = min(len(ds_names_list), len(X_norm))
    ds_names_list = ds_names_list[:min_len]
    X_norm = X_norm[:min_len]

# 获取全量标签
labels_dataset = np.array(ds_names_list)
labels_domain, labels_freq = meta_manager.get_labels(ds_names_list)

print(f"📚 全局数据统计:")
print(f"   - 总样本数: {len(labels_dataset)}")
print(f"   - 包含数据集: {len(np.unique(labels_dataset))} 个")
print(f"   - 包含领域: {np.unique(labels_domain)}") 

print("\n🚀 开始运行 Encoder Benchmark...")

for encoder in encoders:
    print(f"\n⚡ 正在评估 Encoder: {encoder.name}")
    
    # --- 1. 编码所有序列 ---
    embeddings_list = []
    # 使用 DataLoader 批处理 (num_workers=0 避免多进程报错)
    dataset_loader = torch.utils.data.DataLoader(X_norm, batch_size=CONFIG['batch_size'])
    
    for batch in tqdm(dataset_loader, desc="Encoding"):
        emb = encoder.encode(batch)
        embeddings_list.append(emb.cpu().numpy())
        
    embeddings_np = np.concatenate(embeddings_list, axis=0)
    
    # --- 2. 聚类/区分度评估 (带调试信息) ---
    print("   📊 计算聚类指标...")
    
    # 采样逻辑
    total_samples = len(embeddings_np)
    if total_samples > 5000:
        idx = np.random.choice(total_samples, 5000, replace=False)
        emb_subset = embeddings_np[idx]
        lab_ds_subset = labels_dataset[idx]
        lab_dom_subset = labels_domain[idx]
        lab_freq_subset = labels_freq[idx]
    else:
        emb_subset = embeddings_np
        lab_ds_subset = labels_dataset
        lab_dom_subset = labels_domain
        lab_freq_subset = labels_freq
    
    # === 🔍 [新增] 调试：打印当前参与计算的标签分布 ===
    print(f"      [Debug] 当前采样样本数: {len(emb_subset)}")
    
    def print_distribution(name, labels):
        unique, counts = np.unique(labels, return_counts=True)
        dist = dict(zip(unique, counts))
        print(f"      [Debug] {name:<8} : {len(unique)} 类 -> {str(dist)[:100]}..." if len(str(dist))>100 else f"      [Debug] {name:<8} : {len(unique)} 类 -> {dist}")
        return len(unique)

    n_ds = print_distribution("Dataset", lab_ds_subset)
    n_dom = print_distribution("Domain", lab_dom_subset)
    n_freq = print_distribution("Freq", lab_freq_subset)
    # ======================================================

    # 定义安全计算函数 (内部闭包)
    def calc_score(X, labels, n_classes, tag):
        if n_classes < 2:
            print(f"      ⚠️ 跳过 {tag} Silhouette (类别不足 2 个)")
            return np.nan
        try:
            return silhouette_score(X, labels)
        except Exception as e:
            print(f"      ❌ {tag} 计算出错: {e}")
            return np.nan

    # 计算指标
    score_dataset = calc_score(emb_subset, lab_ds_subset, n_ds, "Dataset")
    score_domain = calc_score(emb_subset, lab_dom_subset, n_dom, "Domain")
    score_freq = calc_score(emb_subset, lab_freq_subset, n_freq, "Freq")
    
    print(f"      -> Scores: DS={score_dataset:.3f}, Dom={score_domain:.3f}, Freq={score_freq:.3f}")

    # --- 3. 鲁棒性评估 ---
    print("   🛡️ 计算鲁棒性指标...")
    n_robust = min(200, len(X_norm))
    subset_x = X_norm[:n_robust]
    robust_metrics = evaluate_robustness(encoder, subset_x)
    
    # --- 4. 记录结果 ---
    record = {
        "Encoder": encoder.name,
        "Cluster_Dataset (Sil)": score_dataset,
        "Cluster_Domain (Sil)": score_domain,
        "Cluster_Freq (Sil)": score_freq,
        **robust_metrics 
    }
    results_table.append(record)

# --- 保存与展示 ---
df_results = pd.DataFrame(results_table)
# 简单的 NaN 填充以便显示
print("\n🏆 评估完成！结果如下：")
display(df_results.fillna("N/A")) 

df_results.to_csv(RESULT_CSV, index=False)
print(f"文件已保存至: {RESULT_CSV}")

📚 全局数据统计:
   - 总样本数: 26287
   - 包含数据集: 198 个
   - 包含领域: ['Economic' 'Energy' 'Healthcare' 'Nature' 'Sales' 'Traffic']

🚀 开始运行 Encoder Benchmark...

⚡ 正在评估 Encoder: UniTS


Encoding: 100%|██████████| 411/411 [00:01<00:00, 342.79it/s]


   📊 计算聚类指标...
      [Debug] 当前采样样本数: 5000
      [Debug] Dataset  : 181 类 -> {'GE_LOOP_SEATTLE_5T': 42, 'GE_LOOP_SEATTLE_D': 41, 'GE_LOOP_SEATTLE_H': 38, 'GE_SZ_TAXI_15T': 28, '...
      [Debug] Domain   : 6 类 -> {'Economic': 453, 'Energy': 687, 'Healthcare': 438, 'Nature': 2045, 'Sales': 217, 'Traffic': 1160}
      [Debug] Freq     : 15 类 -> {'10T': 859, '15T': 287, '30T': 173, '4S': 67, '5T': 405, 'A-DEC': 43, 'D': 741, 'H': 1396, 'M': 209...
      -> Scores: DS=-0.213, Dom=-0.055, Freq=-0.141
   🛡️ 计算鲁棒性指标...

⚡ 正在评估 Encoder: TimesFM


Encoding: 100%|██████████| 411/411 [00:16<00:00, 25.42it/s]


   📊 计算聚类指标...
      [Debug] 当前采样样本数: 5000
      [Debug] Dataset  : 181 类 -> {'GE_LOOP_SEATTLE_5T': 35, 'GE_LOOP_SEATTLE_D': 40, 'GE_LOOP_SEATTLE_H': 47, 'GE_SZ_TAXI_15T': 41, '...
      [Debug] Domain   : 6 类 -> {'Economic': 465, 'Energy': 657, 'Healthcare': 425, 'Nature': 2106, 'Sales': 215, 'Traffic': 1132}
      [Debug] Freq     : 15 类 -> {'10T': 898, '15T': 296, '30T': 151, '4S': 69, '5T': 416, 'A-DEC': 34, 'D': 755, 'H': 1387, 'M': 192...
      -> Scores: DS=-0.389, Dom=-0.041, Freq=-0.259
   🛡️ 计算鲁棒性指标...

⚡ 正在评估 Encoder: Moirai


...

   📊 计算聚类指标...
      [Debug] 当前采样样本数: 5000
      [Debug] Dataset  : 180 类 -> {'GE_LOOP_SEATTLE_5T': 37, 'GE_LOOP_SEATTLE_D': 46, 'GE_LOOP_SEATTLE_H': 44, 'GE_SZ_TAXI_15T': 38, '...
      [Debug] Domain   : 6 类 -> {'Economic': 470, 'Energy': 660, 'Healthcare': 418, 'Nature': 2084, 'Sales': 205, 'Traffic': 1163}
      [Debug] Freq     : 15 类 -> {'10T': 882, '15T': 307, '30T': 159, '4S': 84, '5T': 409, 'A-DEC': 32, 'D': 741, 'H': 1431, 'M': 200...
      -> Scores: DS=-0.414, Dom=-0.121, Freq=-0.301
   🛡️ 计算鲁棒性指标...

⚡ 正在评估 Encoder: Wavelet(DWT)


Encoding: 100%|██████████| 411/411 [00:00<00:00, 961.69it/s] 


   📊 计算聚类指标...
      [Debug] 当前采样样本数: 5000
      [Debug] Dataset  : 181 类 -> {'GE_LOOP_SEATTLE_5T': 45, 'GE_LOOP_SEATTLE_D': 28, 'GE_LOOP_SEATTLE_H': 46, 'GE_SZ_TAXI_15T': 43, '...
      [Debug] Domain   : 6 类 -> {'Economic': 471, 'Energy': 653, 'Healthcare': 488, 'Nature': 2056, 'Sales': 201, 'Traffic': 1131}
      [Debug] Freq     : 15 类 -> {'10T': 861, '15T': 299, '30T': 167, '4S': 73, '5T': 442, 'A-DEC': 38, 'D': 734, 'H': 1377, 'M': 197...
      -> Scores: DS=-0.272, Dom=-0.026, Freq=-0.190
   🛡️ 计算鲁棒性指标...

🏆 评估完成！结果如下：


,Encoder,Cluster_Dataset (Sil),Cluster_Domain (Sil),Cluster_Freq (Sil),dist_noise,dist_mask,dist_shift,dist_scale
0,UniTS,-0.212811,-0.054658,-0.140599,0.170105,0.198437,0.258529,0.000000
1,TimesFM,-0.389357,-0.040986,-0.259265,0.014561,0.012510,0.007996,0.000423
2,Moirai,-0.362356,-0.071538,-0.243114,0.013162,0.046654,0.033242,0.000565
3,FFT(Spectral),-0.241102,-0.044754,-0.130919,0.000726,0.017804,0.017939,0.000000
4,Stats(CATCH22-like),-0.413731,-0.121390,-0.300522,0.028967,0.025312,0.025421,0.000131
5,Wavelet(DWT),-0.272444,-0.026041,-0.190436,0.001158,0.044858,0.090677,0.000000


文件已保存至: encoder_benchmark_results.csv
